# CDSE / NASA GIBS realistic Earth pipeline

This notebook downloads the newest **actually published** global true-colour NASA GIBS frame, validates it, saves a web-ready equirectangular Earth texture, and updates `web/public/data/satellite-manifest.json`.

It is designed for Copernicus Data Space Ecosystem JupyterLab after cloning the GitHub repository.

Important: this is near-real-time imagery, not a live video stream. The notebook never invents timestamps. It checks recent UTC dates and uses the first frame that the official service really returns.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import time
from datetime import UTC, datetime, timedelta
from pathlib import Path
from typing import Any
from urllib.parse import urlencode

import requests
from PIL import Image
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "web").exists() and (REPO_ROOT.parent / "web").exists():
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = REPO_ROOT / "web" / "public" / "data" / "satellite"
MANIFEST_PATH = REPO_ROOT / "web" / "public" / "data" / "satellite-manifest.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

WMS_URL = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
LAYER = "VIIRS_SNPP_CorrectedReflectance_TrueColor"
WIDTH = 2048
HEIGHT = 1024
LOOKBACK_DAYS = 14
TIMEOUT_SECONDS = 60

print("Repository:", REPO_ROOT)
print("Output:", OUTPUT_DIR)


In [ ]:
def build_session() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET"}),
        raise_on_status=False,
    )
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Terraforming-Planet/Polar-Sun-Moon-Analysis (scientific demonstrator)"
    })
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session

session = build_session()


In [ ]:
def build_gibs_url(observation_date: str) -> str:
    params = {
        "SERVICE": "WMS",
        "VERSION": "1.3.0",
        "REQUEST": "GetMap",
        "FORMAT": "image/jpeg",
        "TRANSPARENT": "FALSE",
        "LAYERS": LAYER,
        "CRS": "EPSG:4326",
        "STYLES": "",
        "WIDTH": WIDTH,
        "HEIGHT": HEIGHT,
        "BBOX": "-90,-180,90,180",
        "TIME": observation_date,
    }
    return f"{WMS_URL}?{urlencode(params)}"


def validate_image_bytes(data: bytes) -> tuple[int, int]:
    from io import BytesIO
    with Image.open(BytesIO(data)) as image:
        image.verify()
    with Image.open(BytesIO(data)) as image:
        width, height = image.size
        if width < 1024 or height < 512:
            raise ValueError(f"Unexpectedly small image: {width}x{height}")
        return width, height


def download_latest_published_frame() -> dict[str, Any]:
    now = datetime.now(UTC)
    errors: list[str] = []

    for days_back in range(1, LOOKBACK_DAYS + 1):
        candidate = (now - timedelta(days=days_back)).date().isoformat()
        url = build_gibs_url(candidate)
        response = session.get(url, timeout=TIMEOUT_SECONDS)

        content_type = response.headers.get("content-type", "")
        if response.status_code != 200:
            errors.append(f"{candidate}: HTTP {response.status_code}")
            continue
        if not content_type.startswith("image/"):
            errors.append(f"{candidate}: unexpected content type {content_type}")
            continue

        try:
            width, height = validate_image_bytes(response.content)
        except Exception as exc:
            errors.append(f"{candidate}: invalid image ({exc})")
            continue

        filename = f"earth-gibs-viirs-{candidate}.jpg"
        output_path = OUTPUT_DIR / filename
        output_path.write_bytes(response.content)

        checksum = hashlib.sha256(response.content).hexdigest()
        retrieved_at = datetime.now(UTC)
        observation_at = datetime.fromisoformat(candidate).replace(tzinfo=UTC)
        latency_hours = round((retrieved_at - observation_at).total_seconds() / 3600, 2)

        return {
            "timestampUtc": observation_at.isoformat().replace("+00:00", "Z"),
            "retrievedUtc": retrieved_at.isoformat().replace("+00:00", "Z"),
            "officialUrl": url,
            "localPreviewPath": f"data/satellite/{filename}",
            "width": width,
            "height": height,
            "checksumSha256": checksum,
            "contentType": content_type.split(";")[0],
            "latencyHours": latency_hours,
        }

    raise RuntimeError("No valid GIBS frame found. " + " | ".join(errors[-5:]))


frame = download_latest_published_frame()
frame


In [ ]:
def load_manifest() -> dict[str, Any]:
    if not MANIFEST_PATH.exists():
        return {"schemaVersion": 1, "generatedUtc": None, "sources": []}
    return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def upsert_gibs_source(manifest: dict[str, Any], frame: dict[str, Any]) -> dict[str, Any]:
    source = {
        "id": "nasa-gibs-viirs-snpp-true-color",
        "agency": "NASA EOSDIS",
        "satellite": "Suomi NPP",
        "instrument": "VIIRS",
        "coverage": "Global",
        "mode": "near-real-time daily true-colour composite",
        "expectedCadence": "1 day",
        "latestObservationUtc": frame["timestampUtc"],
        "retrievalUtc": frame["retrievedUtc"],
        "processingLatencyHours": frame["latencyHours"],
        "availability": "available",
        "officialSourceUrl": WMS_URL,
        "licenceUsage": "NASA imagery attribution required; verify mission-specific usage guidance",
        "lastError": None,
        "frames": [frame],
    }

    sources = [item for item in manifest.get("sources", []) if item.get("id") != source["id"]]
    sources.append(source)
    sources.sort(key=lambda item: item.get("id", ""))

    manifest["schemaVersion"] = 1
    manifest["generatedUtc"] = datetime.now(UTC).isoformat().replace("+00:00", "Z")
    manifest["sources"] = sources
    return manifest


manifest = upsert_gibs_source(load_manifest(), frame)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Manifest written:", MANIFEST_PATH)
print("Texture written:", REPO_ROOT / "web" / "public" / frame["localPreviewPath"])


In [ ]:
# Optional local validation
saved = REPO_ROOT / "web" / "public" / frame["localPreviewPath"]
assert saved.exists()
assert MANIFEST_PATH.exists()
assert frame["checksumSha256"] == hashlib.sha256(saved.read_bytes()).hexdigest()
assert frame["timestampUtc"] != frame["retrievedUtc"]
print("Validation passed.")


In [ ]:
# Optional Git workflow. Run only after reviewing generated files.
# !git status --short
# !git add web/public/data/satellite-manifest.json web/public/data/satellite/
# !git commit -m "data: update latest NASA GIBS Earth texture"
# !git push origin HEAD
